# Exploratory Data Analysis

This notebook performs a professional exploratory data analysis for the CDC Diabetes Health Indicators dataset. The goal is to assess the dataset structure, understand target balance, and analyze key relationships for a risk prediction workflow.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / 'src'))
from data_loader import load_cdc_diabetes_data

sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})
OUTPUT_DIR = Path.cwd().parent / 'images'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Dataset overview

In [ ]:
X, y, df = load_cdc_diabetes_data()

print('Full dataset shape:', df.shape)
print('Feature matrix shape:', X.shape)
print('Target vector shape:', y.shape)

print('Columns:')
print(df.columns.tolist())

## Prediction task

The dataset is used to build a diabetes risk prediction study based on CDC health indicators and lifestyle variables. This analysis is intended to support data science model development, not medical diagnosis.

In [ ]:
target_name = y.name if y.name else 'Diabetes_binary'

print('Target variable:', target_name)
print('
Target distribution:')
print(y.value_counts(dropna=False))
print('
Relative frequency:')
print(y.value_counts(normalize=True).round(3))

## Missing values and summary statistics

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print('Missing values by column:')
print(missing[missing > 0])

print('Descriptive statistics for numeric columns:')
display(df.describe(include='number'))

## Class imbalance analysis

In [ ]:
plt.figure()
ax = sns.countplot(x=y, palette='deep')
ax.set_title('Diabetes target distribution')
ax.set_xlabel(target_name)
ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom')
plot_path = OUTPUT_DIR / 'target_distribution.png'
plt.tight_layout()
plt.savefig(plot_path)
plt.show()
print(f'Saved plot to {plot_path}')

The target distribution chart identifies whether the dataset is balanced or requires special handling during model development.

## Distribution of key variables

In [ ]:
key_columns = ['BMI', 'Age', 'HighBP', 'HighChol', 'PhysActivity', 'GenHlth', 'Income', 'Education']
available_columns = [col for col in key_columns if col in df.columns]
print('Plotted columns:', available_columns)

for col in available_columns:
    plt.figure()
    if pd.api.types.is_numeric_dtype(df[col]):
        sns.histplot(df[col].dropna(), kde=False, bins=30, color='#2E86AB')
        plt.xlabel(col)
        plt.title(f'Distribution of {col}')
    else:
        sns.countplot(x=col, data=df, palette='deep')
        plt.title(f'Distribution of {col}')
        plt.xticks(rotation=45, ha='right')
    plot_path = OUTPUT_DIR / f'distribution_{col}.png'
    plt.tight_layout()
    plt.savefig(plot_path)
    plt.show()
    print(f'Saved plot for {col} to {plot_path}')

The selected variables are relevant for diabetes risk modeling and cover both clinical indicators and lifestyle measures.

## Relationship between features and diabetes target

In [ ]:
relationship_cols = ['BMI', 'Age', 'GenHlth', 'PhysActivity', 'HighBP', 'HighChol']
relationship_cols = [col for col in relationship_cols if col in df.columns]

for col in relationship_cols:
    if pd.api.types.is_numeric_dtype(df[col]):
        plt.figure()
        sns.boxplot(x=y, y=df[col], palette='muted')
        plt.title(f'{col} by diabetes status')
        plt.xlabel(target_name)
        plt.ylabel(col)
    else:
        plt.figure()
        sns.countplot(x=col, hue=y, data=df, palette='muted')
        plt.title(f'{col} by diabetes status')
        plt.xticks(rotation=45, ha='right')
        plt.xlabel(col)
        plt.ylabel('Count')
    plot_path = OUTPUT_DIR / f'relationship_{col}.png'
    plt.tight_layout()
    plt.savefig(plot_path)
    plt.show()
    print(f'Saved relationship plot for {col} to {plot_path}')

These plots compare the distribution of key variables across diabetes classes, helping to identify features that may be relevant in model training.

## Correlation heatmap

In [ ]:
numeric_df = df.select_dtypes(include=['number'])
corr = numeric_df.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(corr, cmap='coolwarm', center=0, linewidths=0.5)
plt.title('Correlation matrix for numeric features')
plot_path = OUTPUT_DIR / 'correlation_heatmap.png'
plt.tight_layout()
plt.savefig(plot_path)
plt.show()
print(f'Saved correlation heatmap to {plot_path}')

The correlation heatmap provides a high-level view of linear relationships, which is useful for initial feature selection and data understanding.

## Grouped analysis

In [ ]:
group_cols = ['Age', 'BMI', 'GenHlth', 'PhysActivity']
group_cols = [col for col in group_cols if col in df.columns]

for col in group_cols:
    analysis_df = df[[col, target_name]].copy()
    analysis_df = analysis_df.rename(columns={target_name: 'target'})
    if pd.api.types.is_numeric_dtype(analysis_df[col]):
        analysis_df['bucket'] = pd.qcut(analysis_df[col].rank(method='first'), q=5, duplicates='drop')
        summary = analysis_df.groupby('bucket')['target'].mean().reset_index()
        plt.figure()
        sns.lineplot(data=summary, x='bucket', y='target', marker='o')
        plt.xticks(rotation=45, ha='right')
        plt.title(f'Diabetes rate by {col} quintile')
        plt.ylabel('Mean diabetes rate')
    else:
        summary = analysis_df.groupby(col)['target'].mean().reset_index()
        plt.figure()
        sns.barplot(data=summary, x=col, y='target', palette='muted')
        plt.xticks(rotation=45, ha='right')
        plt.title(f'Diabetes rate by {col}')
        plt.ylabel('Mean diabetes rate')
    plot_path = OUTPUT_DIR / f'grouped_{col}.png'
    plt.tight_layout()
    plt.savefig(plot_path)
    plt.show()
    print(f'Saved grouped analysis for {col} to {plot_path}')

Grouped analysis offers a structured view of how diabetes risk varies across age, BMI, general health, and physical activity segments.

## Interpretation summary

This EDA documents the dataset and establishes a foundation for modeling. The focus remains on clear data insights and professional presentation rather than medical conclusions.

# Exploratory Data Analysis

This notebook performs a professional exploratory data analysis for the CDC Diabetes Health Indicators dataset. 
The goal is to assess the dataset structure, understand target balance, and analyze key relationships for a risk prediction workflow.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd().parent / 'src'))
from data_loader import load_cdc_diabetes_data

sns.set_style('whitegrid')
plt.rcParams.update({'figure.figsize': (10, 6), 'figure.dpi': 100})
OUTPUT_DIR = Path.cwd().parent / 'images'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Dataset overview

In [ ]:
X, y, df = load_cdc_diabetes_data()

print('Full dataset shape:', df.shape)
print('Feature matrix shape:', X.shape)
print('Target vector shape:', y.shape)

print('Columns:')
print(df.columns.tolist())

## Prediction task

The dataset is used to build a diabetes risk prediction study based on CDC health indicators and lifestyle variables. 
This analysis is intended to support data science model development, not medical diagnosis.

In [ ]:
target_name = y.name if y.name else 'Diabetes_binary'

print('Target variable:', target_name)
print('
Target distribution:')
print(y.value_counts(dropna=False))
print('
Relative frequency:')
print(y.value_counts(normalize=True).round(3))

## Missing values and summary statistics

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
print('Missing values by column:')
print(missing[missing > 0])

print('Descriptive statistics for numeric columns:')
display(df.describe(include='number'))

## Class imbalance analysis

In [ ]:
plt.figure()
ax = sns.countplot(x=y, palette='deep')
ax.set_title('Diabetes target distribution')
ax.set_xlabel(target_name)
ax.set_ylabel('Count')
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom')
plot_path = OUTPUT_DIR / 'target_distribution.png'
plt.tight_layout()
plt.savefig(plot_path)
plt.show()
print(f'Saved plot to {plot_path}')

The target distribution chart identifies whether the dataset is balanced or requires special handling during model development.

: 
,
: {},
: [

: 
,
: {},
: [
,
,
,
,